# IAD Pipeline — Training
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

This notebook uses the refactored `AnomalyDetectionManager`:
- `train()` / `eval()` no longer take a `tiling` flag or call `adjustPaths()` manually — tiled-ensemble is currently the only supported mode, and paths are resolved internally by `_prepareRun`.
- Before calling an action, we check `manager.can_run(...)` / `manager.get_missing_requirements(...)` so we can show the user *why* something isn't ready yet instead of hitting a bare exception.
- State-related failures raise `ManagerStateError` subclasses (`NoModelLoadedError`, `NoDatasetLoadedError`, `TilingNotConfiguredError`, `ModelNotTrainedError`, `CheckpointNotFoundError`), each carrying a `.missing` list.

## 1. Environment Setup
Configure database URI and API keys.

In [1]:
import os
import sys
import warnings
import yaml
import pathlib


# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'

sys.path.append("src")

import wandb
import logging
from pathlib import Path

from src.manager import AnomalyDetectionManager as ADM
from src.manager import DatasetSession as DS
from src.manager import (
    ManagerState,
    ManagerError,
    ManagerStateError,
    ConfigError,
    NoModelLoadedError,
    NoDatasetLoadedError,
    TilingNotConfiguredError,
    ModelNotTrainedError,
    CheckpointNotFoundError,
)
from src.tiling.tilingCheckpoints import checkTiledCheckpointsExist


warnings.filterwarnings("ignore", category=FutureWarning, module="timm.models.layers")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="openvino.runtime")

logger = logging.getLogger("logger")

wandb.login()

/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
wandb: Currently logged in as: daniel-pommer (daniel-pommer-technische-hochschule-n-rnberg-georg-simon-ohm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Configuration
Set your run parameters here before executing the pipeline.

`ADM.loadProduct(...)` loads the model, configures tiling, and resolves output/checkpoint
paths for you — no manual `adjustPaths()` call needed afterwards.

In [2]:
from src.userConfigs import Product

datasetDir  = Path("datasets/")
configDir   = Path("configs/")
outputPath  = Path("results/")
productPath = Path("Products/cable.yaml")
productConfigPath = Path(configDir / productPath)

product: Product
manager, product = ADM.loadProduct(
    productConfigPath=productConfigPath,
    outputPath=outputPath,
    configDir=configDir,
)

print(f"Loaded product: {product.name}")
print(f"Manager state: {manager.state!r}")

INFO: Initializing Padim model.


/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)
INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Successfully loaded model Padim: Padim(
  (pre_processor): PreProcessor()
  (post_processor): AOIPostProcessor(
    (_image_threshold_metric): F1AdaptiveThreshold()
    (_pixel_threshold_metric): F1AdaptiveThreshold()
    (_image_min_max_metric): MinMax()
    (_pixel_min_max_metric): MinMax()
  )
  (evaluator): Evaluator(
    (val_metrics): ModuleList(
      (0): AUROC()
    )
    (test_metrics): ModuleList(
      (0): AUROC()
      (1): F1Score()
      (2): AUPR()
    )
  )
  (model): PadimModel(
    (feature_extractor): TimmFeatureExtractor(
      (feature_extractor): FeatureListNet(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), 

## 3. Load & Select Dataset

We load the dataset session and select the product's category. `DatasetSession` is
independent of the manager's readiness state — you can load and inspect data before
a model is ready, or vice versa.

In [3]:
datasetSession = DS.loadDatasetFromConfig(product.datasetConfig, overwrite=True, merge=False)
datasetSession.select_category(product.name)

print(f"Dataset: {datasetSession.datasetName}, category: {datasetSession.category}")
print(f"Images in view: {len(datasetSession.FO_Dataset)}")

INFO: Dataset 'MVTecADShort' already exists in database
INFO: Overwriting
INFO: There are 424 images in the MVTecADShort dataset.
INFO: There are 2 categorie(s) in the MVTecADShort dataset.
INFO: ['bottle', 'cable']
INFO: Selected category: cable, 374 images


ERROR: Dataset name 'MVTecADShort-cable' is not available


ERROR: Dataset name 'MVTecADShort-cable' is not available
INFO: Deleting MVTecADShort-cable from database and reloading.
Dataset: MVTecADShort, category: cable
Images in view: 374


## 3.1 Inspect Dataset

In [ ]:
datasetSession.launchSession()

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port
INFO: Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port


INFO: Session: localhost:5151


Dataset:          MVTecADShort-cable
Media type:       image
Num samples:      374
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/


Could not connect session, trying again in 10 seconds



## 4. Readiness Check

Before training, ask the manager what (if anything) is missing, rather than firing the
call and translating an exception. This is the check a UI would run to enable/disable
a "Train" button.

In [5]:
manager.attachDatasetSession(datasetSession)
missing = manager.get_missing_requirements("train")
if missing:
    print("Not ready to train yet. Missing:")
    for item in missing:
        print(f"  - {item}")
else:
    print("Ready to train.")

INFO: Attached dataset 'MVTecADShort' (category=cable)
Ready to train.


## 5. Training

`train()` is tiled-ensemble only for now (the `tiling` parameter has been removed —
non-tiled support will be added later as an explicit branch, once implemented). Paths,
tiling setup, callbacks, and the W&B logger are all handled internally by `_prepareRun`.

We wrap the call so that a `ManagerStateError` (e.g. someone re-running this cell before
`generateModel`/`setupTiling` ran) produces a clear message instead of a raw traceback.

In [6]:
# try:
#     manager.train(
#         trainerConfig=product.trainerConfig,
#         modelConfig=product.modelConfig,
#         datamoduleConfig=product.datamoduleConfig,
#         datasetSession=datasetSession,
#         tilingPipelineConfig=product.tilingPipelineConfig,
#     )
#     print("Training complete.")
# except ManagerStateError as e:
#     print(f"Could not train: {e}")
#     print(f"Missing: {e.missing}")
# except ManagerError as e:
#     print(f"Training failed: {e}")

# print(f"Manager state: {manager.state!r}")

In [7]:
datasetSession.launchSession()
# datasetSession.save()

INFO: Session: localhost:5151


Dataset:          MVTecADShort-cable
Media type:       image
Num samples:      374
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/

## 6. Evaluation

`eval()` requires the model to actually be trained (`ManagerState.TRAINED`), which is
only set once `train()` confirms a checkpoint landed on disk — not just that the call
returned. Check readiness first, same pattern as training.

In [8]:
ckptDir = Path("results/MVTecADShort/cable/Padim/tiled/checkpoints")
missing = manager.get_missing_requirements("eval")
manager.loadCheckpoint(ckptDir, product.tilingPipelineConfig)

# if missing:
#     print("Not ready to evaluate yet. Missing:")
#     for item in missing:
#         print(f"  - {item}")

# if ManagerState.CHECKPOINT_AVAILABLE:
try:
    manager.eval(
        evalConfig=product.trainerConfig,
        modelConfig=product.modelConfig,
        datamoduleConfig=product.datamoduleConfig,
        datasetSession=datasetSession,
        tilingPipelineConfig=product.tilingPipelineConfig,
    )
    print("Evaluation complete.")
except ManagerStateError as e:
    print(f"Could not evaluate: {e}")
except ManagerError as e:
    print(f"Evaluation failed: {e}")

INFO: Attached dataset 'MVTecADShort' (category=cable)
INFO: Tiling configured: TilingPipelineConfig(image_size=(256, 256), tile_size=(140, 140), stride=(128, 128), root_dir=PosixPath('results'), normalization_stage=<NormalizationStage.IMAGE: 'image'>, thresholding_stage=<ThresholdingStage.IMAGE: 'image'>, seam_smoothing=SeamSmoothingConfig(apply=True, sigma=2, width=0.1))
INFO: Dataset used for training: Name:        MVTecADShort-cable
Media type:  image
Num samples: 374
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    anomalyType:      fiftyone.core.fields.EmbeddedDocume

Predict: 0it [00:00, ?it/s]

INFO: Tiled ensemble predicting started using Using ckpt_pathtest data.
INFO: Initializing Padim model.
INFO: Loading pretrained weights from Hugging Face hub (timm/resnet18.a1_in1k)


/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: [timm/resnet18.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO: Missing keys (fc.weight, fc.bias) discovered while loading pretrained weights. This is expected if model is being adapted.
INFO: Loading checkpoint from ckpt_path: results/MVTecADShort/cable/Padim/tiled/checkpoints/model0_0.ckpt. No Model from previous training job available.
INFO: Start of predicting for tile at position (0, 0),


...

INFO: Job VisualizeOnDisk completed successfully.
INFO: Running job 51Visualize



51Visualize: 0it [00:00, ?it/s]

INFO: Starting visualisation for Fiftyone.


51 Visualisation: 100%|██████████| 5/5 [00:01<00:00,  2.95it/s]
51Visualize: 1it [00:01,  1.78s/it]

INFO: Job 51Visualize completed successfully.
Evaluation complete.


## 7. Evaluate on Unknown / Prediction Data

Inference reads its checkpoint from an explicit `trainingDir` (which may belong to a
different manager/session than the one currently in memory) rather than relying on
`self.ckptDir` from the last training run. `inference()` checks that the checkpoint
file actually exists at that path and raises `CheckpointNotFoundError` if not — state
flags alone can't guarantee this, since `trainingDir` is caller-supplied.

This cell is left commented out, matching the placeholder in the original notebook — 
uncomment and adjust `predDatasetName` / `predDatasetDir` to run a prediction pass.

In [9]:
# predDatasetName = "MVTecADShortPred"
# predDatasetDir = datasetDir / predDatasetName
#
# predSession = DS.loadDatasetFromDisk(
#     predDatasetDir,
#     datasetName=predDatasetName,
#     overwrite=True,
#     merge=False,
#     split=("pred",),
# )
# predSession.select_category(product.name)
#
# missing = manager.get_missing_requirements("inference")
# if missing:
#     print("Not ready for inference yet. Missing:")
#     for item in missing:
#         print(f"  - {item}")
# else:
#     try:
#         manager.inference(
#             inferencerConfig=product.inferencerConfig,
#             modelConfig=product.modelConfig,
#             trainingDir=product.modelTrainingDir,
#             datamoduleConfig=product.datamoduleConfig,
#             datasetSession=predSession,
#             tilingPipelineConfig=product.tilingPipelineConfig,
#         )
#         print("Inference complete.")
#     except CheckpointNotFoundError as e:
#         print(f"No checkpoint available: {e}")
#     except ManagerStateError as e:
#         print(f"Could not run inference: {e}")
#     except ManagerError as e:
#         print(f"Inference failed: {e}")
#
# manager.launchSession()